# Assignment 2: Research Agent (Web Search + LLM)

**Goal:** Build a ReAct agent that:
1. Takes a user question
2. Searches the web for the top-3 results (DuckDuckGo)
3. Fetches content from each result
4. Synthesizes a final answer based on all 3 sources

**Approach:** Use `create_react_agent()` with two tools: `internet_search` and `fetch_url`.
The agent autonomously decides when to search, fetch, and answer.

## 1. Setup & Imports

In [1]:
import os
import requests
from dotenv import load_dotenv
from ddgs import DDGS
from bs4 import BeautifulSoup
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

print("Model initialized successfully")

Model initialized successfully


## 2. Define Tools

Two tools for the agent:
- `internet_search` - searches DuckDuckGo and returns top 3 URLs
- `fetch_url` - fetches and cleans text content from a URL

In [2]:
@tool
def internet_search(query: str) -> str:
    """Search DuckDuckGo and return top 3 result URLs."""
    results = DDGS().text(query, region='en-us', max_results=3)
    urls = [r['href'] for r in results]
    return "\n".join(urls) if urls else "No results found."

@tool
def fetch_url(url: str) -> str:
    """Fetch and return cleaned text content from a URL (max 4000 chars)."""
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, timeout=10, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator=' ')
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    clean_text = '\n'.join(chunk for chunk in chunks if chunk)
    return clean_text[:4000]

print("2 tools defined: internet_search, fetch_url")

2 tools defined: internet_search, fetch_url


## 3. Create the Agent

Use `create_react_agent()` with our two tools and a system prompt that instructs the agent to:
1. Search for the query
2. Fetch content from all 3 URLs
3. Synthesize a final answer

In [3]:
agent = create_react_agent(
    model=model,
    tools=[internet_search, fetch_url],
    prompt=(
        "You are a research agent. When the user asks a question, follow these steps:\n"
        "1. Use internet_search to find the top 3 URLs related to the question.\n"
        "2. Use fetch_url on each of the 3 URLs to get their content.\n"
        "3. Synthesize a clear, professional answer based ONLY on the fetched content.\n"
        "Always fetch all 3 URLs before answering. Be concise and cite your sources."
    ),
)

print("Research agent created with internet_search and fetch_url tools")

Research agent created with internet_search and fetch_url tools


C:\Users\Yaz00\AppData\Local\Temp\ipykernel_29276\3955824428.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 4. Run the Agent

In [4]:
response = agent.invoke({
    "messages": [HumanMessage(
        content="What are the benefits of artificial intelligence in healthcare?"
    )]
})

print("=" * 70)
print("FINAL RESEARCH REPORT")
print("=" * 70)
print(response["messages"][-1].content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1773446400000'}, 'provider_name': None}}, 'user_id': 'user_3AN5Knef2j3v9YJNz0i1VLf7Ygs'}